# W&B Reports API — Programmatically Curate a Report

Reports let you pin curated runs, charts, and prose into a single
shareable page — the artifact that turns a scratchpad of experiments
into stakeholder-facing communication. Building them **programmatically**
means the same code can regenerate them after every training pass, so
your dashboards stay in sync with your latest runs by construction.

This notebook builds a report against the runs produced by
`experiment_tracking.ipynb`. Run that notebook first so the target
project contains a `tiny-transolver-train` run to point at.

**Prerequisites**
- `uv sync` (adds `wandb-workspaces` to your environment).
- `uv run wandb login`.
- `experiment_tracking.ipynb` has been executed end-to-end at least
  once, so `wandb_emea/jb_physics_demo` contains a training run.


## 1. Setup

Imports and the target project. `wandb-workspaces` is a separate
package that wraps the Reports REST API; the workspaces module is
imported as `wr` by convention.


In [ ]:
import wandb
import wandb_workspaces.reports.v2 as wr

ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

print(f"wandb            = {wandb.__version__}")
print(f"target project   = {ENTITY}/{PROJECT}")


## 2. Build the report

A `Report` is a title, a description, and an ordered list of `blocks`.
Two kinds of block matter here:

- **Text blocks** (`H1`, `H2`, `P`, `MarkdownBlock`) — static prose
  authored inline, exactly what you would write in a markdown cell.
- **`PanelGrid`** — the live block. It holds one or more `Runset`s
  (each a query into a specific entity/project, optionally filtered by
  run-name substring) and one or more `Panel`s (`LinePlot`, `BarPlot`,
  `ScatterPlot`, `MediaBrowser`, ...). Panels re-query at render time,
  so runs logged **after** you save the report show up automatically.


In [ ]:
panel_grid = wr.PanelGrid(
    runsets=[
        wr.Runset(
            entity=ENTITY,
            project=PROJECT,
            name="TinyTransolver training",
            query="tiny-transolver",           # substring match on run name
        ),
    ],
    panels=[
        wr.LinePlot(
            title="Validation loss decomposition",
            x="epoch",
            y=["val/loss", "val/mse", "val/aux"],
            smoothing_factor=0.3,
        ),
        wr.LinePlot(
            title="Per-channel RMSE (normalised, directly comparable)",
            x="epoch",
            y=["val/pressure_rmse", "val/velocity_rmse"],
        ),
        wr.LinePlot(
            title="Slice-routing health",
            x="epoch",
            y=["val/slice_entropy", "val/slice_usage_entropy"],
        ),
        wr.MediaBrowser(
            media_keys=["model/slice_assignment_epoch_11"],
            num_columns=1,
        ),
    ],
)

blocks = [
    wr.H1(text="TinyTransolver — training summary"),
    wr.P(
        text=(
            "Auto-generated summary of runs produced by "
            "experiment_tracking.ipynb. Panels re-query live at render "
            "time, so new runs appear here without regenerating the report."
        )
    ),
    wr.H2(text="Loss curves and routing health"),
    panel_grid,
    wr.MarkdownBlock(
        text=(
            "**Reading guide.** `val/loss = val/mse + λ·val/aux`. "
            "Per-channel RMSEs are on normalised targets so pressure and "
            "velocity are directly comparable. `val/slice_usage_entropy` "
            "close to `log(M)` indicates healthy slice balance; drops "
            "signal routing collapse."
        )
    ),
]


## 3. Save and share

`report.save()` publishes the report to the target project as a draft.
`report.url` returns the shareable link — the same URL you would get
from clicking **Create report** in the W&B UI.


In [ ]:
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="TinyTransolver — training summary",
    description="Programmatically generated by notebooks/reports.ipynb.",
    blocks=blocks,
)

report.save()

print(report.url)
